# 09 — Example-fit plotting routine

This notebook drives the real `FittingResultPlotter` (`pipelines.param_pipeline.plots`) end to end on a controlled tomogram and inspects the example-fit panels it produces. The plotter selects pixels by R-squared tier and overlays data, total fit, components, and residuals.

Modules exercised:

- `pipelines.param_pipeline.plots.FittingResultPlotter` (real, imported and run)
- `pipelines.param_pipeline.metrics.FittingMetricsCalculator` (real, imported and run)
- `tools.reporting.plotting.PlotBase` (style base of the plotter)

Plots are written to a temp directory and then displayed inline; fixed seed, CPU only.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt

SEED = 0
rng  = np.random.default_rng(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    torch = None

plt.rcParams.update({
    "figure.dpi"     : 110,
    "savefig.dpi"    : 110,
    "font.size"      : 11,
    "axes.grid"      : True,
    "grid.alpha"     : 0.30,
    "grid.linewidth" : 0.4,
})

print("Repo root :", REPO_ROOT)
print("NumPy     :", np.__version__)


In [ ]:
def gaussian_mixture_profile(height_axis, components, noise_std=0.0, rng=None):
    profile = np.zeros_like(height_axis, dtype=np.float64)
    for amp, mu, sigma in components:
        profile += amp * np.exp(-((height_axis - mu) ** 2) / (2.0 * sigma * sigma))
    if noise_std > 0.0:
        draw    = rng.normal(0.0, noise_std, size=height_axis.shape) if rng is not None else np.random.normal(0.0, noise_std, size=height_axis.shape)
        profile = profile + draw
    return profile.astype(np.float32)


def pack_parameters(components_per_pixel, k_max, shape):
    Az, R  = shape
    params = np.zeros((3 * k_max, Az, R), dtype=np.float32)
    for (az, rg), comps in components_per_pixel.items():
        for k, (amp, mu, sigma) in enumerate(comps[:k_max]):
            params[3 * k    , az, rg] = amp
            params[3 * k + 1, az, rg] = mu
            params[3 * k + 2, az, rg] = sigma
    return params


In [ ]:
import tempfile
from pathlib import Path

import matplotlib.image as mpimg

from tools.data.gaussians import GaussianMixture
from pipelines.param_pipeline.metrics import FittingMetricsCalculator
from pipelines.param_pipeline.plots   import FittingResultPlotter
from tools.monitoring.logger import Logger

logger = Logger(log_dir='', name='nb09_plots')
print('plotter imported')

## Controlled tomogram with a spread of fit qualities

We build a two-Gaussian parameter field, generate the matching tomogram, and then deliberately corrupt the stored parameters of a fraction of pixels so the R-squared map spans low, mid, and high tiers. This guarantees the plotter has pixels to draw in every tier.

In [ ]:
K            = 2
Az, R        = 24, 24
H            = 90
height_range = (0.0, 40.0)
height_axis  = np.linspace(*height_range, H).astype(np.float32)

true_params = np.zeros((3 * K, Az, R), dtype=np.float32)
true_params[0] = rng.uniform(0.7, 1.0,  (Az, R))
true_params[1] = rng.uniform(9.0, 13.0, (Az, R))
true_params[2] = rng.uniform(2.0, 3.0,  (Az, R))
true_params[3] = rng.uniform(0.4, 0.7,  (Az, R))
true_params[4] = rng.uniform(24.0, 30.0,(Az, R))
true_params[5] = rng.uniform(2.5, 3.5,  (Az, R))

tomogram = np.zeros((H, Az, R), dtype=np.float32)
for j, h in enumerate(height_axis):
    tomogram[j] = GaussianMixture.evaluate_slice(true_params, float(h), K)
tomogram += rng.normal(0.0, 0.01, tomogram.shape).astype(np.float32)

stored = true_params.copy()
corrupt_mask = rng.random((Az, R)) < 0.4
noise_scale  = rng.uniform(0.0, 6.0, (Az, R)) * corrupt_mask
stored[1] += noise_scale
stored[2] *= (1.0 + 0.5 * noise_scale / 6.0)

tmp_dir  = Path(tempfile.mkdtemp())
tom_path = tmp_dir / 'plot_tomogram.npy'
np.save(tom_path, tomogram)
metadata = {'height_range': list(height_range)}
print('tomogram + stored params ready ->', tmp_dir)

In [ ]:
calc    = FittingMetricsCalculator(n_gaussians=K, logger=logger)
metrics = calc.run(stored, metadata, tom_path)

plotter = FittingResultPlotter(output_directory=tmp_dir, n_gaussians=K, logger=logger, n_fits_per_tier=3)
saved   = plotter.run(stored, metrics, metadata, tom_path)
print('saved figure keys:')
for k in sorted(saved):
    print(' ', k, '->', saved[k].name)

## Display the example-fit tiers

We load the three saved tier images (low, mid, high R-squared) and show them. Each panel overlays the data profile, the dashed total fit, the shaded components, and a residual strip.

In [ ]:
tier_keys = [k for k in ('example_fits_low', 'example_fits_mid', 'example_fits_high') if k in saved]
fig, axes = plt.subplots(len(tier_keys), 1, figsize=(12, 5.5 * len(tier_keys)))
if len(tier_keys) == 1:
    axes = [axes]
for ax, key in zip(axes, tier_keys):
    ax.imshow(mpimg.imread(saved[key]))
    ax.set_axis_off()
    ax.set_title(key)
fig.tight_layout()
plt.show()

## Expected outcome

The plotter should run without error and return the full set of figure keys (colormaps, distributions, metrics, and the three example-fit tiers). In the high-tier panel the dashed fit should sit on the data with a flat residual; in the low-tier panel the fit should visibly miss the data with a large residual, confirming the tier selection and example-fit rendering behave as intended.